# FE10 数值扩展与 NULL 校准

本 Notebook 从当前正式 Feature Baseline **FE10 — `unknown_activity_time`** 出发，记录 NULL / Negative-Control 校准、数值候选目录、第一阶段完整 5-Fold 结果，以及后续 LOO 是否值得开展。Notebook 只读取和分析结果，不负责模型训练。

---

## Table of Contents

1. [实验背景与环境设置](#r2-num-setup)
   - 1.1 [正式 Reference 与目标](#r2-num-contract)
   - 1.2 [导入依赖与定位项目路径](#r2-num-imports)
2. [FE20–FE29 特征目录](#r2-num-catalog)
3. [受控实验协议与运行方式](#r2-num-run)
4. [NULL / Negative-Control 校准](#r2-null-results)
5. [第一阶段数值实验结果](#r2-num-results)
6. [结果分析与决策记录](#r2-num-decisions)
7. [阶段小结与下一步](#r2-num-next)

---

<a id="r2-num-setup" name="r2-num-setup"></a>

## 1. 实验背景与环境设置

<a id="r2-num-contract" name="r2-num-contract"></a>

### 1.1 正式 Reference 与目标

Round 2 不再与原始 BASE 比较，而是统一回答：新增候选在 FE10 之上是否仍有增量。

| 项目 | 结果 |
|---|---:|
| Reference | **FE10 — `unknown_activity_time`** |
| 5-Fold CV AUC | **0.964629 ± 0.000523** |
| Public LB | **0.96606** |
| 核心指标 | `delta_vs_reference` |

正式比较使用同一 Fold 下的配对差值：

```text
AUC(FE10 + candidate) - AUC(FE10)
```

NULL 用于估计 feature-addition noise，不作为正式 p-value。

<a id="r2-num-imports" name="r2-num-imports"></a>

### 1.2 导入依赖与定位项目路径

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.round2_features import ROUND2_FEATURE_INFO

NULL_RESULTS_DIR = PROJECT_ROOT / "results" / "noise_calibration"
ROUND2_RESULTS_DIR = PROJECT_ROOT / "results" / "round2" / "first_stage"
NULL_SUMMARY_PATH = NULL_RESULTS_DIR / "summary.csv"
ROUND2_SUMMARY_PATH = ROUND2_RESULTS_DIR / "summary.csv"

<a id="r2-num-catalog" name="r2-num-catalog"></a>

## 2. FE20–FE29 特征目录

为方便阅读，使用以下缩写：`D` = daily screen，`S` = social，`G` = gaming，`W` = work/study，`WK` = weekend screen。

第一阶段只直接运行 Active 和 Audit。Conditional Reserve 先保留定义，只有同方向 Active 实验成功后才考虑。

In [ ]:
catalog_rows = []

for feature_id in [f"FE{i}" for i in range(20, 30)]:
    definition, status, first_stage_use = ROUND2_FEATURE_INFO[feature_id]
    catalog_rows.append(
        {
            "特征": feature_id,
            "定义": definition,
            "状态": status,
            "第一阶段用途": first_stage_use,
        }
    )

numerical_catalog = pd.DataFrame(catalog_rows)
display(numerical_catalog)

<a id="r2-num-run" name="r2-num-run"></a>

## 3. 受控实验协议与运行方式

所有命令都从项目根目录执行。先检查特征构造；该命令只读取数据和构造特征，不训练 CatBoost：

```bash
python -m src.round2_experiment --validate-only
```

NULL 校准：

```bash
python -m src.noise_calibration
```

第一阶段 numerical 实验按以下位置进入总顺序；O_GROUP 的先验最低，因此要等两个 categorical group 完成后最后运行：

```bash
python -m src.round2_experiment FE27
python -m src.round2_experiment R_GROUP
python -m src.round2_experiment FE26

# 接着运行 06 Notebook 中的 CAT_NUM_GROUP 和 CAT2_GROUP

# 第一阶段最后运行历史 ratio 复审
python -m src.round2_experiment O_GROUP
```

不指定 `--fold` 时自动运行完整 5-Fold；指定 `--fold 1` 可只运行一个 Fold。重复执行相同命令时，兼容的已有 JSON 会被验证并跳过。

<a id="r2-null-results" name="r2-null-results"></a>

## 4. NULL / Negative-Control 校准

NULL-A 是严格冗余列；NULL-B 使用两个固定 seed 对 `daily_screen_time_hours` 做 permutation。三组实验各跑完整 5-Fold，共 15 fits。

In [ ]:
if NULL_SUMMARY_PATH.exists():
    null_summary = pd.read_csv(NULL_SUMMARY_PATH)
    display(null_summary)
else:
    null_summary = pd.DataFrame()
    print("当前还没有 NULL 汇总结果。")

In [ ]:
null_records = []

for path in sorted(NULL_RESULTS_DIR.glob("null_*_fold*.json")):
    null_records.append(json.loads(path.read_text(encoding="utf-8")))

null_folds = pd.DataFrame(null_records)

if null_folds.empty:
    print("当前还没有 NULL 逐折结果。")
else:
    null_pivot = null_folds.pivot(
        index="fold",
        columns="experiment_id",
        values="delta_vs_reference",
    )
    display(null_pivot)

    ax = null_pivot.plot(marker="o", figsize=(9, 5))
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title("NULL Controls: Paired AUC Delta vs FE10")
    ax.set_xlabel("Fold")
    ax.set_ylabel("AUC delta vs FE10")
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

<a id="r2-num-results" name="r2-num-results"></a>

## 5. 第一阶段数值实验结果

本节只查看 FE27、R_GROUP、FE26 与 O_GROUP。CAT_NUM_GROUP 和 CAT2_GROUP 放在 `06_categorical_interactions.ipynb`。

In [ ]:
NUMERICAL_EXPERIMENTS = ["FE27", "R_GROUP", "FE26", "O_GROUP"]

if ROUND2_SUMMARY_PATH.exists():
    round2_summary = pd.read_csv(ROUND2_SUMMARY_PATH)
    numerical_summary = round2_summary.loc[
        round2_summary["experiment_id"].isin(NUMERICAL_EXPERIMENTS)
    ].copy()
    display(numerical_summary)
else:
    round2_summary = pd.DataFrame()
    numerical_summary = pd.DataFrame()
    print("当前还没有 Round 2 第一阶段汇总结果。")

In [ ]:
round2_records = []

for path in sorted(ROUND2_RESULTS_DIR.glob("*_fold*.json")):
    result = json.loads(path.read_text(encoding="utf-8"))
    if result.get("experiment_id") in NUMERICAL_EXPERIMENTS:
        round2_records.append(result)

numerical_folds = pd.DataFrame(round2_records)

if numerical_folds.empty:
    print("当前还没有第一阶段 numerical 逐折结果。")
else:
    numerical_pivot = numerical_folds.pivot(
        index="fold",
        columns="experiment_id",
        values="delta_vs_reference",
    )
    display(numerical_pivot)

    ax = numerical_pivot.plot(marker="o", figsize=(9, 5))
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title("Round 2 Numerical Experiments: Paired AUC Delta")
    ax.set_xlabel("Fold")
    ax.set_ylabel("AUC delta vs FE10")
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

<a id="r2-num-decisions" name="r2-num-decisions"></a>

## 6. 结果分析与决策记录

第一阶段结果与 NULL 都完成后，再根据以下证据记录判断：

| 检查项 | 要回答的问题 |
|---|---|
| Mean delta | 五折平均增量是否为正？ |
| Fold consistency | 正向 Fold 数量和方向是否稳定？ |
| NULL scale | 增量是否明显区别于 feature-addition noise？ |
| Best iteration | 是否存在异常训练轮数漂移？ |
| Complexity | 提升是否足以支持新增特征复杂度？ |

不要在结果不完整时提前填写 Keep / Reject。

<a id="r2-num-next" name="r2-num-next"></a>

## 7. 阶段小结与下一步

- FE27 为最高优先级的单特征实验，不控制其他 Reserve。
- R_GROUP 如果成立，先做 LOO，再考虑 FE21、FE23、FE24。
- FE26 独立代表 leisure vs work/study 轴，不与 R_GROUP 混跑。
- O_GROUP 只复审历史 FE07 / FE16，不重新开启整个 ratio family。
- 本轮 Notebook 不执行 LOO；只有完整结果支持某条支线时，才进入第二阶段。